# Establishing the baseline data and model for Monitoring

In [1]:
import json
import os
import sys

from joblib import dump

rootpath = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(rootpath, "scripts"))
sys.path.append(os.path.join(rootpath, "src"))

In [2]:
from gcp_functions import read_file_as_df, upload_directory
from load_configs import Configs

from data import (
    build_features,
    create_X_y_multistep,
    split_train_test_panel,
)
from models import create_fit_xgbregressor_chain, evaluate_all

configs = Configs()

loading gcs configs for None


In [3]:
f_ref_path = {
    "prefix": "cleaned_samples_prod",
    "fname": "Kaggle_Access_2025-07-22_WSPall_from_2020-07-22.parquet",
}

In [4]:
f_ref_full = f_ref_path["prefix"] + "/" + f_ref_path["fname"]
df_ref = read_file_as_df(
    configs.cloud["gcs"]["project"],
    configs.cloud["gcs"]["data_monitoring_bucket"],
    f_ref_full,
)

In [5]:
df = df_ref.copy()
params = {"CldrFeats": "True", "ModReg": "True", "lags": "3", "steps": "5"}
TARGET = "returns"
SAMPLE_TICKERS = ["AAPL", "AMZN"]

In [6]:
df_train, df_test = split_train_test_panel(df, train_ratio=0.8)
df_train_feats, _features2scale = build_features(
    df_train, lags=int(params["lags"]), split="train", CldrFeats=params["CldrFeats"]
)
df_test_feats, _features2scale = build_features(
    df_test, lags=int(params["lags"]), split="test", CldrFeats=params["CldrFeats"]
)
X_train, y_train = create_X_y_multistep(
    df_train_feats, steps=int(params["steps"]), target=TARGET, split="train"
)
X_test, y_test = create_X_y_multistep(
    df_test_feats, steps=int(params["steps"]), target=TARGET, split="test"
)
# Instantiate and train a model
estimator = create_fit_xgbregressor_chain(X_train, y_train, params["ModReg"])
# Evaluate the model
scores = evaluate_all(estimator, X_train, y_train, X_test, y_test, df, SAMPLE_TICKERS)

In [8]:
X_test.head()

returns_lag1  returns_lag2  returns_lag3  \
Ticker Date                                                            
AAPL   2024-09-10 04:00:00      0.003635     -0.000407      0.007065   
       2024-09-11 04:00:00     -0.011452      0.003635     -0.000407   
       2024-09-12 04:00:00     -0.000494     -0.011452      0.003635   
       2024-09-13 04:00:00      0.001214     -0.000494     -0.011452   
       2024-09-16 04:00:00      0.028569      0.001214     -0.000494   

                                SMA_10      SMA_50      EMA_12      EMA_26  \
Ticker Date                                                                  
AAPL   2024-09-10 04:00:00  224.115002  222.901001  222.954429  222.858745   
       2024-09-11 04:00:00  223.578003  223.019201  222.909133  222.844023   
       2024-09-12 04:00:00  223.206003  223.069201  222.887729  222.838541   
       2024-09-13 04:00:00  222.477003  223.088201  222.828078  222.813464   
       2024-09-16 04:00:00  221.209004  222.887801  221.826836  222.332467   

                                MACD  MACD_Signal  MACD_Hist  ...    BB_STD  \
Ticker Date                                                   ...             
AAPL   2024-09-10 04:00:00  0.095684     0.923116  -0.827432  ...  3.024158   
       2024-09-11 04:00:00  0.065110     0.751515  -0.686405  ...  2.958547   
       2024-09-12 04:00:00  0.049188     0.611049  -0.561861  ...  2.912163   
       2024-09-13 04:00:00  0.014615     0.491762  -0.477148  ...  2.954762   
       2024-09-16 04:00:00 -0.505630     0.292284  -0.797914  ...  3.464640   

                              BB_Upper    BB_Lower  BB_Width     RSI_14  \
Ticker Date                                                               
AAPL   2024-09-10 04:00:00  230.661316  218.564685  0.053855  36.230626   
       2024-09-11 04:00:00  230.599595  218.765406  0.052671  42.718077   
       2024-09-12 04:00:00  230.559326  218.910675  0.051833  46.321077   
       2024-09-13 04:00:00  230.533524  218.714477  0.052617  40.082264   
       2024-09-16 04:00:00  231.066781  217.208220  0.061831  30.411261   

                              ROC_10  sin(1,freq=ME)  cos(1,freq=ME)  \
Ticker Date                                                            
AAPL   2024-09-10 04:00:00 -0.031121    9.510565e-01       -0.309017   
       2024-09-11 04:00:00 -0.023550    8.660254e-01       -0.500000   
       2024-09-12 04:00:00 -0.016425    7.431448e-01       -0.669131   
       2024-09-13 04:00:00 -0.031725    5.877853e-01       -0.809017   
       2024-09-16 04:00:00 -0.055371    1.224647e-16       -1.000000   

                            sin(1,freq=W-SUN)  cos(1,freq=W-SUN)  
Ticker Date                                                       
AAPL   2024-09-10 04:00:00           0.781831           0.623490  
       2024-09-11 04:00:00           0.974928          -0.222521  
       2024-09-12 04:00:00           0.433884          -0.900969  
       2024-09-13 04:00:00          -0.433884          -0.900969  
       2024-09-16 04:00:00           0.000000           1.000000  

[5 rows x 21 columns]

In [9]:
y_test.head()

y_step_1  y_step_2  y_step_3  y_step_4  y_step_5
Ticker Date                                                                 
AAPL   2024-09-10 04:00:00 -0.011452 -0.000494  0.001214  0.028569 -0.002168
       2024-09-11 04:00:00 -0.000494  0.001214  0.028569 -0.002168 -0.017672
       2024-09-12 04:00:00  0.001214  0.028569 -0.002168 -0.017672 -0.035741
       2024-09-13 04:00:00  0.028569 -0.002168 -0.017672 -0.035741  0.002936
       2024-09-16 04:00:00 -0.002168 -0.017672 -0.035741  0.002936  0.007639

In [10]:
y_test_hat = estimator.predict(X_test)

In [15]:
y_test_hat[:, 0].shape

(11722,)

In [16]:
# Evidently cannot deal with multi-output yet, so we need to merge the first step y_test
# merge X_test and first step y_test, and into one dataframe
val_data = X_test.copy()
val_data["target"] = y_test["y_step_1"]
val_data["prediction"] = y_test_hat[:, 0]
val_data.head()

returns_lag1  returns_lag2  returns_lag3  \
Ticker Date                                                            
AAPL   2024-09-10 04:00:00      0.003635     -0.000407      0.007065   
       2024-09-11 04:00:00     -0.011452      0.003635     -0.000407   
       2024-09-12 04:00:00     -0.000494     -0.011452      0.003635   
       2024-09-13 04:00:00      0.001214     -0.000494     -0.011452   
       2024-09-16 04:00:00      0.028569      0.001214     -0.000494   

                                SMA_10      SMA_50      EMA_12      EMA_26  \
Ticker Date                                                                  
AAPL   2024-09-10 04:00:00  224.115002  222.901001  222.954429  222.858745   
       2024-09-11 04:00:00  223.578003  223.019201  222.909133  222.844023   
       2024-09-12 04:00:00  223.206003  223.069201  222.887729  222.838541   
       2024-09-13 04:00:00  222.477003  223.088201  222.828078  222.813464   
       2024-09-16 04:00:00  221.209004  222.887801  221.826836  222.332467   

                                MACD  MACD_Signal  MACD_Hist  ...    BB_Lower  \
Ticker Date                                                   ...               
AAPL   2024-09-10 04:00:00  0.095684     0.923116  -0.827432  ...  218.564685   
       2024-09-11 04:00:00  0.065110     0.751515  -0.686405  ...  218.765406   
       2024-09-12 04:00:00  0.049188     0.611049  -0.561861  ...  218.910675   
       2024-09-13 04:00:00  0.014615     0.491762  -0.477148  ...  218.714477   
       2024-09-16 04:00:00 -0.505630     0.292284  -0.797914  ...  217.208220   

                            BB_Width     RSI_14    ROC_10  sin(1,freq=ME)  \
Ticker Date                                                                 
AAPL   2024-09-10 04:00:00  0.053855  36.230626 -0.031121    9.510565e-01   
       2024-09-11 04:00:00  0.052671  42.718077 -0.023550    8.660254e-01   
       2024-09-12 04:00:00  0.051833  46.321077 -0.016425    7.431448e-01   
       2024-09-13 04:00:00  0.052617  40.082264 -0.031725    5.877853e-01   
       2024-09-16 04:00:00  0.061831  30.411261 -0.055371    1.224647e-16   

                            cos(1,freq=ME)  sin(1,freq=W-SUN)  \
Ticker Date                                                     
AAPL   2024-09-10 04:00:00       -0.309017           0.781831   
       2024-09-11 04:00:00       -0.500000           0.974928   
       2024-09-12 04:00:00       -0.669131           0.433884   
       2024-09-13 04:00:00       -0.809017          -0.433884   
       2024-09-16 04:00:00       -1.000000           0.000000   

                            cos(1,freq=W-SUN)    target  prediction  
Ticker Date                                                          
AAPL   2024-09-10 04:00:00           0.623490 -0.011452   -0.000418  
       2024-09-11 04:00:00          -0.222521 -0.000494    0.001958  
       2024-09-12 04:00:00          -0.900969  0.001214    0.000237  
       2024-09-13 04:00:00          -0.900969  0.028569    0.000317  
       2024-09-16 04:00:00           1.000000 -0.002168    0.001086  

[5 rows x 23 columns]

In [17]:
ref_path = os.path.join(rootpath, "ref_data_model")

os.makedirs(ref_path, exist_ok=True)

# save params as json
with open(os.path.join(ref_path, "ref_params.json"), "w") as f:
    json.dump(params, f)
# save scores as json
with open(os.path.join(ref_path, "ref_scores.json"), "w") as f:
    json.dump(scores, f)
# save model as joblib bin
with open(os.path.join(ref_path, "model.bin"), "wb") as f:
    dump(estimator, f)
# save data as parquet
val_data.to_parquet(os.path.join(ref_path, "val_data.parquet"))

In [18]:
upload_directory(
    project_id=configs.cloud["gcs"]["project"],
    bucket_name=configs.cloud["gcs"]["data_monitoring_bucket"],
    folder="ref_data_model",
    local_dir=ref_path,
)

Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/ref_params.json to gs://stocks-forecasting-data-monitoring/ref_data_model/ref_params.json
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/model.bin to gs://stocks-forecasting-data-monitoring/ref_data_model/model.bin
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/ref_scores.json to gs://stocks-forecasting-data-monitoring/ref_data_model/ref_scores.json
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/val_data.parquet to gs://stocks-forecasting-data-monitoring/ref_data_model/val_data.parquet
Successfully uploaded 4 file(s) to gs://stocks-forecasting-data-monitoring/ref_data_model
